# 05 · Replication — Copies That Disagree

**Part 1 · Data-Intensive Applications** · *Prerequisites: 01, 03* · *Book mapping: DDIA ch. 5*

We keep copies of data on multiple machines for three reasons: **availability** (survive
failures — nb 00's redundancy math), **latency** (data near users — nb 01's WAN numbers), and
**read throughput** (many replicas serve many readers). If data never changed, replication
would be `cp`. All difficulty comes from **change**: every write must reach every copy, over a
network that delays, drops, and reorders messages — while readers are looking.

Three architectures answer "who may write?", and this notebook builds all three on our
discrete-event simulator (`sysdes.sim`):

1. **Single-leader**: one node takes all writes and streams them to followers.
2. **Multi-leader**: several nodes take writes and exchange them — hello, conflicts.
3. **Leaderless**: *clients* write to several replicas directly; reads vote (quorums).

**Learning objectives.**

1. Build log-shipping replication and demonstrate why *asynchronous* replication can lose
   acknowledged writes on failover.
2. Reproduce the three replication-lag anomalies — read-your-writes, monotonic reads,
   consistent prefix — and implement the standard fixes.
3. Demonstrate multi-leader write conflicts; show why last-write-wins silently destroys data
   and what keeping siblings buys instead.
4. Implement Dynamo-style quorum reads/writes with read repair; verify the R+W>N overlap rule
   — and construct the counterexample showing quorums still aren't linearizable.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from sysdes import viz
from sysdes.sim import Network, Node, Sim

viz.use_style()

## 1. Single-leader replication

> All writes go to the **leader**, which appends them to a **replication log** (a sequence of
> changes, each with a **log sequence number, LSN**) and streams the log to **followers**,
> which apply changes *in LSN order*. Reads may go to any replica. This is Postgres, MySQL,
> MongoDB, Kafka partitions — the default architecture of the industry.

In [ ]:
fig, ax = viz.system_diagram(
    nodes=[
        {"id": "c", "x": 0, "y": 1, "label": "clients", "kind": "client"},
        {"id": "L", "x": 2.6, "y": 1, "label": "leader", "kind": "store"},
        {"id": "f1", "x": 5.2, "y": 1.9, "label": "follower 1", "kind": "store"},
        {"id": "f2", "x": 5.2, "y": 0.1, "label": "follower 2", "kind": "store"},
    ],
    edges=[
        {"src": "c", "dst": "L", "label": "all writes"},
        {"src": "L", "dst": "f1", "label": "log (LSN 1,2,3...)"},
        {"src": "L", "dst": "f2", "label": "log"},
        {"src": "c", "dst": "f2", "label": "reads", "style": "dashed", "curve": 0.45},
    ],
    title="Single-leader: one write home, many read homes",
)
plt.show()

In [ ]:
class Leader(Node):
    """Accepts writes, assigns LSNs, streams the log to followers.

    sync_count = how many follower acknowledgements must arrive before the
    client's write is acknowledged: 0 = fully asynchronous (fast, risky),
    1 = semi-synchronous (one follower guaranteed current), etc.
    """

    def __init__(self, name, follower_names, sync_count=0):
        super().__init__(name)
        self.store: dict[str, object] = {}
        self.log: list[tuple[int, str, object]] = []
        self.followers = follower_names
        self.sync_count = sync_count
        self.acked_lsns: list[int] = []       # writes CONFIRMED to the client
        self._pending: dict[int, int] = {}    # lsn -> follower acks so far

    def client_write(self, key, value) -> int:
        lsn = len(self.log) + 1
        self.log.append((lsn, key, value))
        self.store[key] = value               # leader applies immediately
        for f in self.followers:
            self.send(f, {"type": "replicate", "lsn": lsn, "key": key, "value": value})
        if self.sync_count == 0:
            self.acked_lsns.append(lsn)       # async: ack BEFORE followers have it
        else:
            self._pending[lsn] = 0
        return lsn

    def handle(self, src, msg):
        if msg["type"] == "ack":
            lsn = msg["lsn"]
            if lsn in self._pending:
                self._pending[lsn] += 1
                if self._pending[lsn] >= self.sync_count:
                    self.acked_lsns.append(lsn)   # enough copies exist: safe to ack
                    del self._pending[lsn]


class Follower(Node):
    """Applies the log strictly in LSN order.

    The network reorders messages (random latency), so LSN 5 can arrive before
    LSN 4 - we buffer out-of-order entries until the gap fills. Skipping ahead
    would make the replica's state a sequence of changes that never happened.
    """

    def __init__(self, name):
        super().__init__(name)
        self.store: dict[str, object] = {}
        self.applied_lsn = 0
        self._buffer: dict[int, tuple[str, object]] = {}

    def handle(self, src, msg):
        if msg["type"] == "replicate":
            self._buffer[msg["lsn"]] = (msg["key"], msg["value"])
            while self.applied_lsn + 1 in self._buffer:   # apply any ready prefix
                self.applied_lsn += 1
                key, value = self._buffer.pop(self.applied_lsn)
                self.store[key] = value
                self.send(src, {"type": "ack", "lsn": self.applied_lsn})

def build_cluster(sync_count=0, latency=(2.0, 12.0), seed=0):
    sim = Sim()
    net = Network(sim, latency=latency, seed=seed)
    leader = Leader("L", ["F1", "F2"], sync_count=sync_count)
    f1, f2 = Follower("F1"), Follower("F2")
    for n in (leader, f1, f2):
        net.add(n)
    return sim, net, leader, f1, f2

# Sanity: writes flow leader -> followers, in order, despite reordering.
sim, net, leader, f1, f2 = build_cluster()
for i in range(20):
    sim.at(float(i), leader.client_write, f"k{i}", i)
sim.run()
assert f1.store == leader.store == f2.store, "followers converge to the leader's state"
assert f1.applied_lsn == 20
print(f"20 writes replicated; followers applied all {f1.applied_lsn} LSNs in order")

### 1.1 Failure first: asynchronous replication loses acknowledged writes

With `sync_count=0` the client is told "saved!" the moment the leader has the write. If the
leader dies before the log ships — and a **failover** promotes a follower — the write is gone.
Not delayed: *gone*, after the user saw a success message.

In [ ]:
fig, ax = viz.message_timeline(
    actors=["client", "leader", "follower"],
    messages=[
        {"src": "client", "dst": "leader", "send": 0, "recv": 1, "label": "write x=1"},
        {"src": "leader", "dst": "client", "send": 1, "recv": 2, "label": "ACK: saved!",
         "color": "#2e7d32"},
        {"src": "leader", "dst": "follower", "send": 1, "recv": 6,
         "label": "replicate x=1", "lost": True},
    ],
    events=[{"t": 3, "actor": "leader", "label": " crash"},
            {"t": 7, "actor": "follower", "label": " promoted: x missing"}],
    title="Async replication: acked, then lost",
)
plt.show()

In [ ]:
# The same story in the simulator:
sim, net, leader, f1, f2 = build_cluster(sync_count=0, seed=1)
sim.at(0.0, leader.client_write, "post:9", "my great post")
# Leader crashes at t=0.5 - before the replicate messages (latency >= 2) land.
# We model the crash as a partition; in-flight messages to the dead node's
# peers are swallowed at delivery time.
sim.at(0.5, net.partition, {"L"}, {"F1", "F2"})
sim.run()

assert 1 in leader.acked_lsns, "the client was told the write succeeded"
assert "post:9" not in f1.store and "post:9" not in f2.store
promoted = f1                       # failover: F1 is the new leader
print(f"acked write present on promoted node? {'post:9' in promoted.store} "
      "-> an ACKNOWLEDGED write vanished")

# Semi-synchronous (sync_count=1): ack only after >=1 follower has the write.
sim, net, leader, f1, f2 = build_cluster(sync_count=1, seed=1)
sim.at(0.0, leader.client_write, "post:9", "my great post")
sim.at(0.5, net.partition, {"L"}, {"F1", "F2"})
sim.run()
assert leader.acked_lsns == [], "no follower ack arrived -> client was NEVER told 'saved'"
print("semi-sync: the same crash produces a visible error, not silent loss")

That is the entire sync/async trade in two runs: **async buys write latency and availability
(a slow follower never blocks writes) by making "saved" a probabilistic statement.** Real
systems mix modes: one synchronous follower for durability, the rest async (exercise 1
quantifies what that buys and what it doesn't).

### 1.2 Replication lag: three anomalies and their fixes

With async replication, followers run behind by the **replication lag**. Eventually every
follower catches up ("**eventual consistency**") — but "eventually" is where users live.
First, let's measure the anomaly window:

In [ ]:
def stale_read_probability(read_delay: float, trials: int = 150) -> float:
    """Write at t=0, read follower F1 at t=read_delay. Stale = value not there yet."""
    stale = 0
    results = []
    for trial in range(trials):
        sim, net, leader, f1, f2 = build_cluster(seed=1000 + trial)
        sim.at(0.0, leader.client_write, "x", "new")
        sim.at(read_delay, lambda: results.append("x" in f1.store))
        sim.run()
    return 1 - float(np.mean(results))

delays = [0.5, 2, 4, 6, 8, 10, 13]
p_stale = [stale_read_probability(d) for d in delays]

fig, ax = plt.subplots()
ax.plot(delays, p_stale, "o-")
ax.set_xlabel("time between write and read (same units as network latency 2-12)")
ax.set_ylabel("P(read sees stale data)")
ax.set_title("The anomaly window IS the replication lag distribution")
plt.show()

assert p_stale[0] == 1.0, "reading right after a write: guaranteed stale (latency >= 2)"
assert p_stale[-1] == 0.0, "after max latency, followers have converged"
assert all(a >= b for a, b in zip(p_stale, p_stale[1:])), "staleness decays with delay"

**Anomaly 1 — reading your own writes.** You comment, the page reloads from a lagging
follower, your comment is gone. Users forgive stale views of *other* people's data; they do
not forgive losing *their own*. The guarantee to engineer: **read-your-writes consistency**.
Standard fix: the client remembers the LSN of its last write; a replica may serve that
client's read only if it has applied at least that LSN (else: wait, or bounce to the leader).

In [ ]:
sim, net, leader, f1, f2 = build_cluster(seed=7)

read_log = []
def read_naive(follower, key, label):
    read_log.append((label, follower.store.get(key)))

def read_with_lsn(follower, key, min_lsn, label):
    """The fix: refuse to serve a read from a replica that hasn't caught up
    to the reader's own last write."""
    if follower.applied_lsn >= min_lsn:
        read_log.append((label, follower.store.get(key)))
    else:
        read_log.append((label, "NOT_CAUGHT_UP -> retry/leader"))

sim.at(0.0, lambda: my_lsn.append(leader.client_write("bio", "I love replication")))
my_lsn: list[int] = []
sim.at(1.0, read_naive, f1, "bio", "naive read at t=1")
sim.at(1.0, lambda: read_with_lsn(f1, "bio", my_lsn[0], "LSN-guarded read at t=1"))
sim.at(30.0, lambda: read_with_lsn(f1, "bio", my_lsn[0], "LSN-guarded read at t=30"))
sim.run()

for label, result in read_log:
    print(f"{label:28} -> {result}")
assert read_log[0][1] is None, "naive read: the user's own write is invisible"
assert read_log[1][1] == "NOT_CAUGHT_UP -> retry/leader", "guard refuses to lie"
assert read_log[2][1] == "I love replication", "once caught up, the follower serves"

**Anomaly 2 — monotonic reads.** Two refreshes hit two followers with different lags: the
comment appears, then *disappears* — time ran backwards. Guarantee: **monotonic reads** (never
observe older state after newer). Fix: pin each user to one replica (e.g., route by
`hash(user_id)` — and notice the fix is a *routing* policy, not a storage change):

In [ ]:
sim, net, leader, f1, f2 = build_cluster(seed=3)
# Make F2 badly lagged: its link is cut while the write replicates to F1 only.
sim.at(0.0, net.cut, "L", "F2")
sim.at(1.0, leader.client_write, "comment:1", "nice post!")
seen = []
sim.at(20.0, lambda: seen.append(("read#1 from F1", f1.store.get("comment:1"))))
sim.at(21.0, lambda: seen.append(("read#2 from F2", f2.store.get("comment:1"))))
sim.run()

print(seen)
assert seen[0][1] == "nice post!" and seen[1][1] is None, \
    "second read went BACK IN TIME - monotonic reads violated"
# The fix in one line: both reads to hash-chosen replica -> same replica -> monotone.
replica_for_user = lambda user_id: [f1, f2][hash(f"u{user_id}") % 2]
assert replica_for_user(1) is replica_for_user(1), "sticky routing"
print("fix: pin user -> replica (sticky routing); staleness allowed, regression not")

**Anomaly 3 — consistent prefix.** With data split across independently-replicating
partitions (nb 06), an observer can see an *answer* (fast partition) before its *question*
(slow partition) — causality inverted for the reader, even though each partition applied its
own writes in order:

In [ ]:
# Two independent leader->follower chains = two partitions of the data.
sim = Sim()
net = Network(sim, latency=(2.0, 4.0), seed=5)
qa_leader, qa_follower = Leader("QL", ["QF"]), Follower("QF")   # questions
an_leader, an_follower = Leader("AL", ["AF"]), Follower("AF")   # answers
for n in (qa_leader, qa_follower, an_leader, an_follower):
    net.add(n)

sim.at(0.0, net.cut, "QL", "QF")            # the question partition lags badly...
sim.at(1.0, qa_leader.client_write, "q7", "What causes replication lag?")
sim.at(2.0, an_leader.client_write, "a7", "Asynchrony. (re: q7)")   # ...answer races ahead
observed = []
sim.at(10.0, lambda: observed.append(
    {"question": qa_follower.store.get("q7"), "answer": an_follower.store.get("a7")}))
sim.run()

print("observer at t=10 sees:", observed[0])
assert observed[0]["answer"] is not None and observed[0]["question"] is None, \
    "the answer is visible before the question it replies to"

The guarantee violated is **consistent prefix reads**: observers should see writes in an
order that *some* single history could produce. The cure needs cross-partition ordering —
exactly the machinery of nb 09 (causality, total order) — or keeping causally-related data in
one partition.

## 2. Multi-leader replication

Run a leader **per datacenter** (or per device, for offline apps; per user, for collaborative
editors): writes are fast and local, and each leader asynchronously forwards its writes to
the others. The price is fundamental: **two leaders can accept conflicting writes for the
same key at the same moment**. No protocol prevents it — the conflict already happened when
each leader acked its client. All you choose is *how to resolve*.

In [ ]:
class MultiLeader(Node):
    """A leader that also applies writes forwarded by its peer.

    mode='lww':      keep the single version with the highest timestamp
                     (Last-Write-Wins - converges by DESTROYING one write).
    mode='siblings': keep every version not causally superseded; the
                     application must merge later (Dynamo/Riak style).
    """

    def __init__(self, name, peer, mode="lww"):
        super().__init__(name)
        self.peer = peer
        self.mode = mode
        # key -> set of (ts, value); LWW keeps the set at size 1.
        self.data: dict[str, set] = {}

    def client_write(self, key, value):
        ts = (self.sim.now, self.name)               # unique, tie-broken timestamp
        seen = self.data.get(key, set())             # versions this write supersedes
        self._apply(key, ts, value, supersedes={s[0] for s in seen})
        self.send(self.peer, {"type": "repl", "key": key, "ts": ts, "value": value,
                              "supersedes": {s[0] for s in seen}})

    def handle(self, src, msg):
        if msg["type"] == "repl":
            self._apply(msg["key"], msg["ts"], msg["value"], msg["supersedes"])

    def _apply(self, key, ts, value, supersedes):
        current = self.data.get(key, set())
        if self.mode == "lww":
            # Highest timestamp wins, everything else is discarded. Silent!
            keep = max(current | {(ts, value)}, key=lambda sv: sv[0])
            self.data[key] = {keep}
        else:
            # Drop versions this write causally saw; keep true concurrents.
            remaining = {sv for sv in current if sv[0] not in supersedes}
            self.data[key] = remaining | {(ts, value)}

def concurrent_edit(mode):
    sim = Sim()
    net = Network(sim, latency=(5.0, 8.0), seed=11)
    a = MultiLeader("A", "B", mode)
    b = MultiLeader("B", "A", mode)
    net.add(a), net.add(b)
    # Two users, two datacenters, editing the same wiki title concurrently -
    # each local leader ACKS before hearing about the other write.
    sim.at(1.0, a.client_write, "title", "Replication For Fun")
    sim.at(1.5, b.client_write, "title", "Replication For Profit")
    sim.run()
    return a, b

a, b = concurrent_edit("lww")
assert a.data["title"] == b.data["title"], "LWW converges on both leaders..."
(winner,) = a.data["title"]
print(f"LWW result on both leaders: {winner[1]!r}")
assert winner[1] == "Replication For Profit", "...but the earlier acked write is GONE"

a, b = concurrent_edit("siblings")
assert a.data["title"] == b.data["title"] and len(a.data["title"]) == 2, \
    "siblings mode converges too - keeping BOTH versions for the app to merge"
print(f"siblings result: {sorted(v for _, v in a.data['title'])}")

Both modes *converge* (all replicas end identical — the minimum bar, called **eventual
consistency**). The difference is what convergence costs: **LWW converges by silently
discarding an acknowledged write** — and in nb 08 we'll see clock skew means the "last" write
may not even be the later one. **Siblings preserve both** and push the merge to the
application (shopping carts union their items; text editors need smarter merges; CRDTs —
data structures whose merges are automatic — are the systematized version). Conflict
*avoidance* is the third option: route all writes for a given key through one home leader,
which is just single-leader per key — partitioning, nb 06.

## 3. Leaderless replication (Dynamo-style quorums)

Remove the leader entirely: the client (or a coordinator) writes to **all N replicas** and
declares success after **W** acknowledge; reads query all and take the newest among the first
**R** answers (every value carries a version). If

$$W + R > N$$

then any read set overlaps any write set in at least one replica — the read *must* see the
newest write. Availability falls out too: with N=3, W=R=2, any single replica can be down and
both reads and writes still succeed. No failover machinery at all.

In [ ]:
class Replica(Node):
    """Stores (version, value) per key; applies a write only if newer."""

    def __init__(self, name):
        super().__init__(name)
        self.data: dict[str, tuple[int, object]] = {}

    def handle(self, src, msg):
        if msg["type"] == "write":
            current = self.data.get(msg["key"], (0, None))
            if msg["version"] > current[0]:
                self.data[msg["key"]] = (msg["version"], msg["value"])
            self.send(src, {"type": "write_ack", "rid": msg["rid"]})
        elif msg["type"] == "read":
            version, value = self.data.get(msg["key"], (0, None))
            self.send(src, {"type": "read_reply", "rid": msg["rid"],
                            "replica": self.name, "version": version, "value": value})


class Coordinator(Node):
    """Fires quorum operations and collects replies (in arrival order)."""

    def __init__(self, name):
        super().__init__(name)
        self.replies: dict[str, list[dict]] = {}

    def write(self, rid, replicas, key, value, version):
        self.replies[rid] = []
        for r in replicas:
            self.send(r, {"type": "write", "rid": rid, "key": key,
                          "value": value, "version": version})

    def read(self, rid, replicas, key):
        self.replies[rid] = []
        for r in replicas:
            self.send(r, {"type": "read", "rid": rid, "key": key})

    def handle(self, src, msg):
        self.replies[msg["rid"]].append(msg)

    def read_result(self, rid, r_quorum):
        """Newest value among the first R replies (None if quorum not met)."""
        replies = self.replies[rid][:r_quorum]
        if len(replies) < r_quorum:
            return None
        return max(replies, key=lambda m: m["version"])

def quorum_cluster(seed=0, latency=(1.0, 6.0)):
    sim = Sim()
    net = Network(sim, latency=latency, seed=seed)
    coord = Coordinator("C")
    replicas = [Replica(f"R{i}") for i in range(3)]
    for n in [coord, *replicas]:
        net.add(n)
    return sim, net, coord, [r.name for r in replicas], {r.name: r for r in replicas}

# N=3, W=2, R=2 - one replica down, service continues:
sim, net, coord, names, reps = quorum_cluster(seed=2)
sim.at(0.0, net.partition, {"C", "R0", "R1"}, {"R2"})          # R2 is unreachable
sim.at(1.0, coord.write, "w1", names, "cart", ["book"], 1)
sim.at(30.0, coord.read, "r1", names, "cart")
sim.run()

write_acks = len(coord.replies["w1"])
result = coord.read_result("r1", r_quorum=2)
print(f"write acks: {write_acks}/3 (W=2 met) -> read returns {result['value']}")
assert write_acks == 2 and result["value"] == ["book"], \
    "one dead replica: cluster still fully available - no failover needed"

### 3.1 Read repair — how stale replicas heal

R2 was down during the write; it holds nothing. Quorum systems heal lazily: a read that
observes disagreeing replicas **writes the newest version back** to the stale ones.

In [ ]:
sim.at(60.0, net.heal)
sim.at(61.0, coord.read, "r2", names, "cart")
sim.run()

stale = [m for m in coord.replies["r2"] if m["version"] == 0]
newest = max(coord.replies["r2"], key=lambda m: m["version"])
print(f"replicas answering with stale version: {[m['replica'] for m in stale]}")
assert any(m["replica"] == "R2" for m in stale), "R2 missed the write while down"

# Read repair: push the newest version to whoever answered stale.
def read_repair():
    for m in stale:
        coord.send(m["replica"], {"type": "write", "rid": "repair",
                                  "key": "cart", "value": newest["value"],
                                  "version": newest["version"]})
coord.replies["repair"] = []
sim.at(80.0, read_repair)
sim.run()
assert reps["R2"].data["cart"] == (1, ["book"]), "read repair healed the stale replica"
print("read repair complete: all 3 replicas identical again")

(The complementary mechanism, **anti-entropy**, is a background process comparing replicas
directly — repair for data nobody happens to read. Real systems run both. And when a write's
home replicas are unreachable, **sloppy quorums** accept writes on *stand-in* nodes with a
note to forward later — **hinted handoff** — trading the overlap guarantee for availability;
exercise 5 makes you find the hole that opens.)

### 3.2 Failure first: why R+W>N still isn't linearizability

The overlap rule guarantees a read *touches* a replica with the newest version. It does
**not** make the system behave like a single copy. While a write is only partially applied,
two readers — both obeying R=2 — can disagree, *in the wrong order*:

In [ ]:
sim, net, coord, names, reps = quorum_cluster(seed=8, latency=(1.0, 1.0))
sim.at(0.0, coord.write, "w0", names, "x", "v1", 1)      # baseline: everyone has v1
# The v2 write reaches R0 instantly, but its messages to R1, R2 crawl:
sim.at(10.0, coord.write, "w2", ["R0"], "x", "v2", 2)
sim.at(50.0, coord.write, "w2b", ["R1", "R2"], "x", "v2", 2)

# Reader ALICE (t=20) happens to query {R0, R1}; reader BOB (t=30) queries {R1, R2}.
sim.at(20.0, coord.read, "alice", ["R0", "R1"], "x")
sim.at(30.0, coord.read, "bob", ["R1", "R2"], "x")
sim.run()

alice = coord.read_result("alice", 2)["value"]
bob = coord.read_result("bob", 2)["value"]
print(f"Alice (starts first) sees {alice!r}; Bob (starts AFTER Alice finished) sees {bob!r}")
assert alice == "v2" and bob == "v1", \
    "Bob read after Alice yet saw the older value - not linearizable"

Alice saw the new value; Bob, starting strictly later, saw the old one. A system acting like
a single copy could never do that. Quorums give you durable, available, *eventually
consistent* storage with bounded staleness — but "acts like one copy" (**linearizability**)
requires more machinery: notebook 09. Meanwhile, notice what versions did here: our integer
versions ordered writes because one coordinator issued them; with many writers you need
**version vectors** to even tell "newer" from "concurrent" — also nb 09.

## 4. Choosing an architecture

| | Single-leader | Multi-leader | Leaderless |
|---|---|---|---|
| Write conflicts | none (one writer) | **yes — must resolve** | yes (concurrent versions) |
| Write latency (multi-region) | one region pays WAN | local everywhere | tunable via W |
| Failure handling | failover (fraught: nb 08) | per-leader independence | none needed (quorums absorb) |
| Consistency ceiling | can be strong (sync/consensus) | eventual + merges | eventual + repair |
| Operational feel | simple, mature | subtle (conflict debt) | simple writes, subtle tuning |
| Home turf | the default; OLTP | multi-region writes, offline apps, co-editing | high-availability KV (carts, sessions) |

Rules of thumb: start single-leader — it's the least machinery for the most guarantee. Go
multi-leader only when *writes* must be local in several regions (or offline), and budget
real engineering for merges. Go leaderless when availability and latency dominate and the
data merges naturally (sets, carts, counters).

## 5. Exercises

1. **Semi-sync accounting.** With 1 synchronous + 1 asynchronous follower, exactly which
   simultaneous failures lose acknowledged writes? What does upgrading to `sync_count=2` cost
   during a follower's slow disk episode (think nb 01's queues)?
2. **Cross-device read-your-writes.** Your LSN fix stores `last_lsn` in the writing client.
   The user posts from their phone and refreshes on their laptop. Design two fixes (one
   client-side, one server-side) and their costs.
3. **Cart merge.** Two offline devices add and remove items from one shopping cart, then
   sync. Show a concrete interleaving where LWW loses an *addition*, and design sibling
   metadata that merges correctly for adds AND removes (careful: a naive union resurrects
   removed items).
4. **Quorum tuning.** N=5. Choose (W, R) for: (a) a write-heavy metrics store where reads may
   be slightly stale; (b) a read-heavy config store where reads must see the latest accepted
   write. What does (W=1, R=5) give and break?
5. **The sloppy hole.** With sloppy quorums + hinted handoff (N=3, W=2), construct a timeline
   where a client's acknowledged write is invisible to a subsequent R=2 read even though
   R+W>N. Which guarantee did sloppiness actually trade away?

### Solutions

**1.** Loss requires losing the leader **and** the one synchronous follower before the async
follower catches up — two simultaneous failures (or one, if they share a rack: nb 00's
correlated faults!). `sync_count=2` survives that but now *every* write waits for the slower
follower: its p99 becomes your write p99, and one follower's slow disk (queueing, nb 01)
stalls the entire write path — availability traded for durability.

**2.** Client-side: return the LSN token to the client and **sync it across devices** via
whatever channel they share (the account) — cheap, but the laptop must fetch the token before
reading. Server-side: track `user → last_write_lsn` in a (replicated…) session service and
gate reads on it — transparent to clients, but adds a lookup to every read and its own
consistency problem. Real systems often approximate: route a user's reads to the leader for
N seconds after any write.

**3.** LWW: phone adds "boots" at ts 10; laptop removes "socks" at ts 11; if the *cart* is
one LWW value, laptop's version (no boots) wins — the addition is gone. Correct merge: track
per-item **add/remove sets with timestamps** (an OR-set CRDT): merge unions both sets; an
item is in the cart if its latest add beats its latest remove. Naive union of cart *contents*
instead resurrects "socks" — removals must be first-class facts, which is tombstones (nb 03)
all over again.

**4.** (a) W=1 or 2, R=4 or 5 — cheap fast writes, reads pay; but at W=1 a single replica
crash right after an ack can lose the write, so metrics tolerance decides. (b) W=4, R=2 (or
W=3,R=3): overlap with slack, reads cheap-ish and fresh. (W=1, R=5): fastest possible writes
and reads that see everything — but *any* replica down blocks **all** reads: you bought
freshness with total read fragility.

**5.** R0,R1 unreachable from the writer's side of a partition → sloppy quorum writes to
{R2, S (stand-in)} → W=2 met, client acked. Partition heals slowly; S hasn't forwarded its
hint yet. Reader queries {R0, R1} → R=2 met → sees the old version. R+W>N held numerically,
but the write set wasn't the *home* set — sloppiness trades away the **overlap guarantee**
itself, keeping only durability (the write exists... somewhere).

## 6. Takeaways

- Replication = shipping a **log of changes**; every architecture is a policy about who may
  append to it.
- **Async replication**: fast, available — and "saved" becomes probabilistic; failover can
  discard acknowledged writes. Know your lag; the anomaly window *is* the lag distribution.
- The lag anomalies have named guarantees and cheap fixes: **read-your-writes** (LSN gating),
  **monotonic reads** (sticky routing), **consistent prefix** (co-partition causal data) —
  choose guarantees per feature, not one global setting.
- **Conflicts are inherent** to multiple writers. LWW converges by destroying data; siblings
  converge by making the app merge; avoidance routes each key to one writer.
- **Quorums** (R+W>N) give availability without failover and freshness-by-overlap — but not
  single-copy behavior. Reads heal the cluster (read repair); versions need vectors once
  writers multiply.

**Further reading.** DDIA ch. 5; the Dynamo paper (SOSP'07) — quorums, hinted handoff,
vector clocks in production; Shapiro et al., *CRDTs: Consistency without concurrency
control* (2011).

**Next:** [06 · Partitioning](06-partitioning.ipynb) — replication copies *all* the data;
partitioning splits it. Every big system does both at once.